<a href="https://colab.research.google.com/github/simongydvandy/CS-4262-Doodling-Project/blob/main/4262%20Downloading%20Script.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cell 1 — Setup & config (run first, always)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import requests
import json
import os
import random

SAMPLES_PER_CAT = 2000
SEED            = 42
DRIVE_BASE      = "/content/drive/MyDrive/quickdraw-classifier/"

RAW_NPY    = DRIVE_BASE + "data/raw/npy/"
RAW_NDJSON = DRIVE_BASE + "data/raw/ndjson/"
PROCESSED  = DRIVE_BASE + "data/processed/"

for path in [RAW_NPY, RAW_NDJSON, PROCESSED]:
    os.makedirs(path, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)

CATEGORIES_URL = "https://raw.githubusercontent.com/googlecreativelab/quickdraw-dataset/master/categories.txt"
NPY_BASE       = "https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap/"
NDJSON_BASE    = "https://storage.googleapis.com/quickdraw_dataset/full/simplified/"

r = requests.get(CATEGORIES_URL)
categories = [line.strip() for line in r.text.strip().split("\n")]
print(f"Found {len(categories)} categories")

with open(DRIVE_BASE + "data/categories.txt", "w") as f:
    f.write("\n".join(categories))
print("Saved categories.txt")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found 345 categories
Saved categories.txt


# Cell 2 — Numpy bitmaps download (~15–25 min)

In [ ]:
print("--- Downloading numpy bitmaps (CNN path) ---")
X_cnn, y_cnn = [], []
failed_npy = []

for idx, cat in enumerate(categories):
    url_name   = cat.replace(" ", "%20") + ".npy"
    local_path = RAW_NPY + cat.replace(" ", "_") + ".npy"

    if not os.path.exists(local_path):
        try:
            resp = requests.get(NPY_BASE + url_name, timeout=30)
            resp.raise_for_status()
            with open(local_path, "wb") as f:
                f.write(resp.content)
            print(f"  [{idx+1}/{len(categories)}] Downloaded {cat}")
        except Exception as e:
            print(f"  [{idx+1}/{len(categories)}] FAILED {cat}: {e}")
            failed_npy.append(cat)
            continue
    else:
        print(f"  [{idx+1}/{len(categories)}] Cached   {cat}")

    # Load data using memory-mapping to avoid OOM for very large files
    data = np.load(local_path, mmap_mode='r')

    # Get number of available samples and select up to SAMPLES_PER_CAT
    total_samples_in_file = len(data)
    num_samples_to_select = min(SAMPLES_PER_CAT, total_samples_in_file)

    if total_samples_in_file > 0:
        # Randomly select indices without loading the entire array into RAM
        random_indices = np.random.choice(total_samples_in_file, num_samples_to_select, replace=False)
        selected_data = data[random_indices]
    else:
        selected_data = np.array([]) # Handle empty files

    X_cnn.append(selected_data)
    y_cnn.append(np.full(num_samples_to_select, idx))
    del selected_data  # free RAM before next iteration

X_cnn = np.concatenate(X_cnn)
y_cnn = np.concatenate(y_cnn)
perm  = np.random.permutation(len(X_cnn))
X_cnn, y_cnn = X_cnn[perm], y_cnn[perm]

np.save(PROCESSED + "X_cnn.npy", X_cnn)
np.save(PROCESSED + "y_cnn.npy", y_cnn)

print(f"\nCNN data saved — X: {X_cnn.shape}, y: {y_cnn.shape}")
if failed_npy:
    print(f"Failed: {failed_npy}")

--- Downloading numpy bitmaps (CNN path) ---
  [1/345] Downloaded aircraft carrier
  [2/345] Downloaded airplane
  [3/345] Downloaded alarm clock
  [4/345] Downloaded ambulance
  [5/345] Downloaded angel
  [6/345] Downloaded animal migration
  [7/345] Downloaded ant
  [8/345] Downloaded anvil
  [9/345] Downloaded apple
  [10/345] Downloaded arm
  [11/345] Downloaded asparagus
  [12/345] Downloaded axe
  [13/345] Downloaded backpack
  [14/345] Downloaded banana
  [15/345] Downloaded bandage
  [16/345] Downloaded barn
  [17/345] Downloaded baseball
  [18/345] Downloaded baseball bat
  [19/345] Downloaded basket
  [20/345] Downloaded basketball
  [21/345] Downloaded bat
  [22/345] Downloaded bathtub
  [23/345] Downloaded beach
  [24/345] Downloaded bear
  [25/345] Downloaded beard
  [26/345] Downloaded bed
  [27/345] Downloaded bee
  [28/345] Downloaded belt
  [29/345] Downloaded bench
  [30/345] Downloaded bicycle
  [31/345] Downloaded binoculars
  [32/345] Downloaded bird
  [33/345] Dow

# Cell 3 — ndjson download (~2–3 hrs, run overnight)

In [ ]:
print("--- Downloading simplified ndjson (stroke feature path) ---")
print("Tip: keep this tab open and laptop plugged in.\n")
records       = []
failed_ndjson = []

for idx, cat in enumerate(categories):
    url_name   = cat.replace(" ", "%20") + ".ndjson"
    local_path = RAW_NDJSON + cat.replace(" ", "_") + ".ndjson"

    if not os.path.exists(local_path):
        try:
            resp = requests.get(NDJSON_BASE + url_name, stream=True, timeout=60)
            resp.raise_for_status()
            with open(local_path, "wb") as f:
                for chunk in resp.iter_content(chunk_size=8192):
                    f.write(chunk)
            print(f"  [{idx+1}/{len(categories)}] Downloaded {cat}")
        except Exception as e:
            print(f"  [{idx+1}/{len(categories)}] FAILED {cat}: {e}")
            failed_ndjson.append(cat)
            continue
    else:
        print(f"  [{idx+1}/{len(categories)}] Cached   {cat}")

    with open(local_path, "r") as f:
        lines = f.readlines()

    random.shuffle(lines)
    for line in lines[:SAMPLES_PER_CAT]:
        obj = json.loads(line)
        records.append({
            "category":   cat,
            "label":      idx,
            "recognized": obj["recognized"],
            "drawing":    obj["drawing"]
        })

random.shuffle(records)
with open(PROCESSED + "strokes.ndjson", "w") as f:
    for rec in records:
        f.write(json.dumps(rec) + "\n")

print(f"\nStroke data saved — {len(records)} records")
if failed_ndjson:
    print(f"Failed: {failed_ndjson}")